In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.window import Window

In [0]:
df = spark.table("ecommerce_lakehouse.bronze.reviews").select("*")

In [0]:
df.count()

In [0]:
df.limit(5).display()

In [0]:
# df_orders = spark.table("ecommerce_lakehouse.silver.orders").select("*")

In [0]:
# orphan_reviews = df.join(df_orders, on="order_id", how="left_anti")
# print(f"reviews with no matching order: {orphan_reviews.count():,}")

### CASTING

In [0]:
# df = df.withColumn("review_score", col("review_score").cast("int"))

In [0]:
df.printSchema()

### DeDup

In [0]:
# w = Window.partitionBy(col("review_id")).orderBy(col("_load_timestamp").desc())
w = Window.partitionBy("review_id").orderBy(
    col("review_answer_timestamp").desc(),   # latest review activity
    col("review_creation_date").desc()       # tiebreaker
)
df = df.withColumn("rn", row_number().over(w)).filter(col("rn") == 1).drop("rn")

In [0]:
print("bronze.reviews:", spark.table("ecommerce_lakehouse.bronze.reviews").count())
# after dedup:
print("silver.reviews:", df.count())

### Null Check

In [0]:
df = df.filter(col("review_id").isNotNull())

In [0]:
df.count()

In [0]:
df.display()

In [0]:
df = df.drop("_corrupt_record")

### Silver table Write

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS ecommerce_lakehouse.silver")

In [0]:
df.write\
    .format("delta")\
        .mode("overwrite")\
            .option("inferSchema", True)\
                .option("overwriteSchema", True)\
                    .saveAsTable("ecommerce_lakehouse.silver.reviews")

In [0]:
spark.table("ecommerce_lakehouse.silver.reviews").count()

In [0]:
spark.table("ecommerce_lakehouse.silver.reviews").limit(5).display()